# מפגש 22 — מחברת הדגמה מבוקרת
Pipeline, Stratified CV, Penguins סנפיר ומסה, Seed 42. יש צורך בגישה ל־CSV.

In [ ]:
import numpy as np
assert np.isclose(np.mean([0.8,0.9,0.7]),0.8)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score
df = pd.read_csv("https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv")
assert len(df)==344
X = df[["flipper_length_mm","body_mass_g"]].copy()
y = df["species"].copy()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42,stratify=y)
assert len(X_train)==258 and len(X_test)==86

בכל קיפול fit חדש של השלמה, תקנון ומודל על האימון הפנימי בלבד; test_score הוא Validation פנימי.

In [ ]:
candidates=[1,3,5,15,51]
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
rows=[]
for k in candidates:
    pipe=make_pipeline(SimpleImputer(strategy="mean"),StandardScaler(),KNeighborsClassifier(n_neighbors=k))
    scores=cross_validate(pipe,X_train,y_train,cv=cv,scoring="f1_macro",return_train_score=True,n_jobs=1)
    assert len(scores["test_score"])==5
    rows.append({"K":k,"train_mean":scores["train_score"].mean(),"cv_mean":scores["test_score"].mean(),"cv_std":scores["test_score"].std(ddof=0)})
results_df=pd.DataFrame(rows)
assert len(results_df)==5
print(results_df)

In [ ]:
best_k=int(results_df.sort_values(["cv_mean","K"],ascending=[False,True]).iloc[0]["K"])
final_model=make_pipeline(SimpleImputer(strategy="mean"),StandardScaler(),KNeighborsClassifier(n_neighbors=best_k))
final_model.fit(X_train,y_train)
predictions=final_model.predict(X_test)
assert best_k in candidates and len(predictions)==86
print("K",best_k,"macro-F1",f1_score(y_test,predictions,average="macro"),"Accuracy",accuracy_score(y_test,predictions))

מדוע אי אפשר לשנות K אחרי שראינו את Test?